# Experiment 0: second-campaign checks

Debugging evidence only: every pilot audit, the exact-resume, FP32-capacity and evaluation checks, monitor timing and the LGD density audit. Nothing here is a scientific result.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, campaign2 as c2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('experiment0/01_campaign2_checks')
report = cp.NotebookReport('Experiment 0: second-campaign checks', sink=sink)

## 1. Pilot audits and GPU checks

Pass/fail per pilot and per check task.

In [2]:
summary = c2.experiment0_summary()
tables = c2.experiment0_tables(summary)
print('Experiment 0 summary available:', summary is not None)
for name, frame in tables.items():
    report.table(name, frame)
report.add('1. Pilot audits and GPU checks', 'Passed overall: ' + str(None if summary is None else summary.get('passed')))

Experiment 0 summary available: False


## 2. LGD density roughness

Median adjacent-bucket log ratio of each original TabPFN regressor per LGD table.

In [3]:
import pandas as pd
density = (summary or {}).get('density_audit') or {}
report.table('Density roughness', pd.DataFrame(density.get('rows', [])))
report.add('2. LGD density roughness', 'Threshold 2.5; the window objective makes the comb harmless for training.')

""


## 3. Training precision against FP32

Question: does a mixed-precision gradient at update zero point where the FP32 gradient points? Agreeing losses do not show this.

In [4]:
from src.visualize import results2 as r2
agreement = c2.precision_agreement(summary)
fig = r2.precision_heatmap(agreement, title='Mixed-precision gradients against FP32 at the pretrained weights')
cp.show(sink, c2.pages(fig, 'precision_agreement',
    'Cosine between the gradient of the training loss computed under mixed precision and under FP32, at the '
    'pretrained weights (update zero) on the identical episode of 8,192 rows, per task and base model (rows) '
    'and for each model\'s largest training tables (columns, largest first). Left: BF16 autocast (TabPFN only). '
    'Right: FP16 autocast with loss scaling; TabICLv2 keeps its column embedder in FP32. A value of 1 means '
    'the same update direction; empty cells could not be computed in FP32 memory.'))
report.table('Update-zero loss and gradient agreement with FP32', agreement)
report.add('3. Training precision against FP32', c2.evidence_note('precision check', agreement))

No measurements available for this section yet.


""


## Summary

Every displayed table and figure, in section order.

In [5]:
print(report.summary(sink))

Experiment 0: second-campaign checks

1. Pilot audits and GPU checks
Passed overall: None

2. LGD density roughness
Threshold 2.5; the window objective makes the comb harmless for training.

Table: Density roughness
CSV (first column: row index)
row

3. Training precision against FP32
No results yet for precision check; the figure is drawn once they exist.

No measurements available for this figure group.

Table: Update-zero loss and gradient agreement with FP32
CSV (first column: row index)
row

experiment0/01_campaign2_checks: no figures saved.
